<div style="border: 1px solid #eee; padding: 15px; border-radius: 5px; text-align: center;">
  <h1 style="font-size: 2em; margin-bottom: 5px;">Nordeus Data Engineering Challenge 2026</h1>
  <p style="font-size: 1.2em; color: #555; margin-top: 5px;">Author: Djordje Mirosavic</p>
</div>

TODO list
- [x] Data Loading
- [x] Cleaning registration required fields
- [x] Cleaning session_ping required fields
- [x] Checking types of each fields
- [x] Checking valid values for `state`, `outcome`,...
- [x] Checking rows with duplicated ids
- [x] Checking rows with duplicated everything else despite ids
- [x] Reconstructing Nan data from user_id using match_start and match_finish
- [x] Recovering user_id from match finish data

# Imports

In [75]:
import sys
import os
import json
import pandas as pd
import numpy as np
from rich import print
import sqlite3
import requests

from google.colab import files

import plotly.express as px
from datetime import datetime, timedelta

from typing import List, Dict, NewType
import threading
from flask import Flask, request, jsonify

# Data loading (upload `events` and `maps` files)

In [2]:
def load_jsonl_to_dataframe(file_path: str) -> pd.DataFrame:
  """
  Loads a .jsonl file from the given path into a pandas DataFrame.

  Args:
    file_path (str): The full path to the .jsonl file.

  Returns:
    pd.DataFrame: A DataFrame containing the loaded data, or None if an error occurs.
  """
  data = []
  try:
    with open(file_path, 'r') as f:
      for line in f:
        data.append(json.loads(line))

    print(f"Successfully loaded {len(data)} events from '{os.path.basename(file_path)}'.")

    df = pd.DataFrame(data)
    print("Data loaded into DataFrame.")
    return df

  except FileNotFoundError:
    print(f"Error: The file '{os.path.basename(file_path)}' was not found at '{file_path}'. Please check the file name and path.")
    return None

  except json.JSONDecodeError as e:
    print(f"Error decoding JSON from '{os.path.basename(file_path)}': {e}")
    return None

  except Exception as e:
    print(f"An unexpected error occurred: {e}")
    return None

In [3]:
def process_jsonl_file(file_path: str) -> pd.DataFrame:
    """
    Loads a .jsonl file into a DataFrame using 'load_jsonl_to_dataframe'
    and prints relevant information.
    """
    file_name = os.path.basename(file_path)
    print(f"Fajl '{file_name}' je uspešno uploadovan.")

    df = load_jsonl_to_dataframe(file_path)

    if df is not None:
      print(f"DataFrame za '{file_name}' je uspešno kreiran.")
      print("Prvih 5 redova novog DataFrame-a:")
      print(df.head())
    else:
      print(f"Neuspešno kreiranje DataFrame-a za fajl '{file_name}'.")

    return df

In [4]:
def upload_jsonl_file(file_name: str) -> pd.DataFrame:
  print(f"Izaberite {file_name} fajl")
  uploaded_files = files.upload()

  if uploaded_files:
    # Assuming only one file is intended to be processed based on original code's 'break'.
    filename = list(uploaded_files.keys())[0]
    df = process_jsonl_file(filename)
    return df
  else:
    print("Nijedan fajl nije izabran")
    return None

In [5]:
events_df = upload_jsonl_file(file_name="events.jsonl")

Izaberite events.jsonl fajl

Saving events.jsonl to events.jsonl


Fajl 'events.jsonl' je uspešno uploadovan.

Successfully loaded 5000 events from 'events.jsonl'.

Data loaded into DataFrame.

DataFrame za 'events.jsonl' je uspešno kreiran.

Prvih 5 redova novog DataFrame-a:

id   timestamp    event_type                               user_id  \
0  2790  1775171101  registration  5129c6fc-f112-4675-ad99-af28f058a59c   
1  2791  1775174844  session_ping  5129c6fc-f112-4675-ad99-af28f058a59c   
2  2792  1775174964  session_ping  5129c6fc-f112-4675-ad99-af28f058a59c   
3  2793  1775175084  session_ping  5129c6fc-f112-4675-ad99-af28f058a59c   
4  2794  1775175204  session_ping  5129c6fc-f112-4675-ad99-af28f058a59c   

                                          event_data  
0  {'country': 'DEU', 'device_os': 'iOS', 'userna...  
1       {'state': 'started', 'device_os': 'Android'}  
2   {'state': 'in_progress', 'device_os': 'Android'}  
3   {'state': 'in_progress', 'device_os': 'Android'}  
4   {'state': 'in_progress', 'device_os': 'Android'}

In [6]:
maps_df = upload_jsonl_file(file_name="maps.jsonl")

Izaberite maps.jsonl fajl

Saving maps.jsonl to maps.jsonl


Fajl 'maps.jsonl' je uspešno uploadovan.

Successfully loaded 5 events from 'maps.jsonl'.

Data loaded into DataFrame.

DataFrame za 'maps.jsonl' je uspešno kreiran.

Prvih 5 redova novog DataFrame-a:

id         name
0  55d20063-c89a-42c4-932d-e1af17936576         Lake
1  fa508820-49dd-472a-b3bb-88931d7c059f  Cobblestone
2  db6f0747-d360-4427-be0c-82b78e51f707      Inferno
3  89e9ef59-d7df-459a-b558-52b00d5b8f91       Desert
4  42093495-255a-47b5-bdcd-347451b9f3bf       Forest

## Data cleaning

## Checking type of each field

In [7]:
'''
  Check data types and valid values within 'event_data' for a given event_type
'''
def check_nested_field_types_and_values(df_filtered_by_event_type: pd.DataFrame,
                                        event_type_name: str,
                                        fields_and_expected_python_types: dict[str, str]):
  print(f"\n--- Checking 'event_data' fields for event_type: '{event_type_name}' ---")

  if df_filtered_by_event_type.empty:
    print(f"No events found for event_type: '{event_type_name}'.")
    return

  for field_name, expected_python_type_str in fields_and_expected_python_types.items():
    # Type check
    actual_types = df_filtered_by_event_type['event_data'].apply(
      lambda x: type(x.get(field_name)).__name__ if isinstance(x, dict) and field_name in x else 'missing_or_not_dict'
    )
    existing_types = actual_types[actual_types != 'missing_or_not_dict']

    if not existing_types.empty:
      unique_actual_types = existing_types.unique()
      if len(unique_actual_types) > 1 or (unique_actual_types[0] != expected_python_type_str):
        print(f"  Field '{field_name}': Mismatched type. Expected '{expected_python_type_str}', found '{unique_actual_types}'.")
      else:
        print(f"  Field '{field_name}': Type check passed (expected '{expected_python_type_str}', found '{unique_actual_types[0]}').")
    else:
      print(f"  Field '{field_name}': Not found or 'event_data' is not a dictionary for any '{event_type_name}' events.")

    # Valid values check
    if field_name in valid_values:
      actual_values = df_filtered_by_event_type['event_data'].apply(
        lambda x: x.get(field_name) if isinstance(x, dict) and field_name in x else None
      )
      present_values = actual_values.dropna() # Drop None values before checking validity
      invalid_value_count = present_values[~present_values.isin(valid_values[field_name])].count()

      if invalid_value_count > 0:
        print(f"  Field '{field_name}': Found {invalid_value_count} invalid values out of {len(present_values)} present values. Expected one of {valid_values[field_name]}.")
      else:
        print(f"  Field '{field_name}': All present values are valid (from {valid_values[field_name]}).")
    else:
      print(f"  Field '{field_name}': No specific valid values to check.")


#### Checking types in events_df

##### Checking shared columns

In [8]:
print("Checking columns data types in events_df:")
column_type_dict = {
  'id': 'int64',
  'timestamp': 'int64',
  'event_type': 'object', # Pandas dtype for string
  'user_id': 'object' # Pandas dtype for string
}
for col, expected_type in column_type_dict.items():
  if str(events_df[col].dtype) == expected_type:
    print(f"Column '{col}': Type is correct.")
  else:
    print(f"Column '{col}': Mismatched type. Expected '{expected_type}', found '{events_df[col].dtype}'.")

Checking columns data types in events_df:

Column 'id': Type is correct.

Column 'timestamp': Type is correct.

Column 'event_type': Type is correct.

Column 'user_id': Type is correct.

#### Checking valid values in event_data

In [9]:
# Define valid values for fields within 'event_data' and for 'event_type'
valid_values = {
  'event_type': ["registration", "session_ping", "match_start", "match_finish"],
  'state': ["started", "in_progress", "ended"],
  'device_os': ["iOS", "Android"],
  'outcome': [1.0, 0.5, 0.0]
}

# Check valid values for 'event_type' column
print("\n--- Checking 'event_type' valid values ---")
invalid_event_types_count = events_df[~events_df['event_type'].isin(valid_values['event_type'])].shape[0]
if invalid_event_types_count > 0:
  print(f"  Found {invalid_event_types_count} rows with invalid 'event_type' values. Expected one of {valid_values['event_type']}.")
else:
  print(f"  All 'event_type' values are valid (from {valid_values['event_type']}).")

--- Checking 'event_type' valid values ---

All 'event_type' values are valid (from ['registration', 'session_ping', 'match_start', 'match_finish']).

##### Checking specific columns based on `event_type`

In [10]:
# Filter DataFrame by event_type for nested checks
events_registration_df = events_df[events_df['event_type'] == 'registration'].copy()
events_session_ping_df = events_df[events_df['event_type'] == 'session_ping'].copy()
events_match_start_df = events_df[events_df['event_type'] == 'match_start'].copy()
events_match_finish_df = events_df[events_df['event_type'] == 'match_finish'].copy()

# Perform nested checks for each event type
check_nested_field_types_and_values(events_registration_df, 'registration', {
    'country': 'str',
    'device_os': 'str',
    'username': 'str'
})

check_nested_field_types_and_values(events_session_ping_df, 'session_ping', {
    'state': 'str',
    'device_os': 'str'
})

check_nested_field_types_and_values(events_match_start_df, 'match_start', {
    'map_id': 'str',
    'opponent_id': 'str'
})

check_nested_field_types_and_values(events_match_finish_df, 'match_finish', {
    'map_id': 'str',
    'opponent_id': 'str',
    'outcome': 'float'
})

--- Checking 'event_data' fields for event_type: 'registration' ---

Field 'country': Type check passed (expected 'str', found 'str').

Field 'country': No specific valid values to check.

Field 'device_os': Type check passed (expected 'str', found 'str').

Field 'device_os': All present values are valid (from ['iOS', 'Android']).

...

Field 'map_id': No specific valid values to check.

Field 'opponent_id': Type check passed (expected 'str', found 'str').

Field 'opponent_id': No specific valid values to check.

Field 'outcome': Type check passed (expected 'float', found 'float').

Field 'outcome': All present values are valid (from [1.0, 0.5, 0.0]).

## Checking duplicates

In [11]:
events_df_orig = events_df.copy()

ako je timestamp striktno rastuci, onda mozemo i da prosto zadrzimo prvi podatak. Tj. ako vidimo da su 2 id-ja ista, zadrzimo prvi jer znamo da se desio ranije

In [12]:
(events_df['timestamp'].diff()[1:] < 0).any()
# Ovo govori da je niz neopadajuci, sto nam odgovara da sme da zadrzi prvi element u duplikatu

np.False_

In [13]:
print(f"Unique ids: {len(events_df['id'].unique())}")

Unique ids: 4890

### Removing `id` duplicates

In [14]:
# Identify duplicate rows across all columns, excluding the 'event_data' column
duplicated_rows = events_df.drop(columns=['event_data']).duplicated(keep=False)
print(f"Number of duplicate rows: {duplicated_rows.sum()}")

initial_rows = len(events_df)
events_df.drop_duplicates(subset=['id'], keep='first', inplace=True)
rows_removed = initial_rows - len(events_df)
print(f"Removed {rows_removed} duplicates.")

Number of duplicate rows: 136

Removed 110 duplicates.

Uklanjamo duplikate gde moze da se desi da su svi podaci isti osim `id` sto nije validno

### Removing elements with all same data except `id`

In [15]:
# gledamo sve kolone osim id i event_data, i ako su sve iste znaci da su i podaci verovatno
cols_to_consider = [col for col in events_df.columns if col not in ['event_data', 'id']]

initial_rows = len(events_df)
events_df.drop_duplicates(subset=cols_to_consider, keep='first', inplace=True)
rows_removed = initial_rows - len(events_df)

print(f"Removed {rows_removed} duplicate rows from events_df, keeping the first occurrence.")
print(f"New events_df shape: {events_df.shape}")

Removed 10 duplicate rows from events_df, keeping the first occurrence.

New events_df shape: (4880, 5)

## Reconstructing NaN values

Trying to connect NaN values with existing data

In [16]:
events_df_with_nan = events_df[events_df.isnull().any(axis=1)]
print("DataFrame 'events_df_with_nan' created with rows containing NaN values:")
print(events_df_with_nan.head(2))

DataFrame 'events_df_with_nan' created with rows containing NaN values:

id   timestamp    event_type user_id  \
122  2024  1775221976  session_ping     NaN   
257  2819  1775249518  session_ping     NaN   

                                           event_data  
122  {'state': 'in_progress', 'device_os': 'Android'}  
257  {'state': 'in_progress', 'device_os': 'Android'}

In [17]:
# Izvuci sve match_finish i match_start
finish_df = events_df[events_df['event_type'] == 'match_finish'].copy()
start_df = events_df[events_df['event_type'] == 'match_start'].copy()

for df in [finish_df, start_df]:
  df['opponent_id'] = df['event_data'].apply(lambda x: x.get('opponent_id'))
  df['map_id'] = df['event_data'].apply(lambda x: x.get('map_id'))

finish_df['outcome'] = finish_df['event_data'].apply(lambda x: x.get('outcome'))

Gledamo match_finish slucaj gde je `user_id`=NaN, i ako nadjemo partnera sa kojim je zavrsio partiju, onda mozemo da rekonstruisemo `user_id`

### Recovering `user_id` from `match_start` and `match_finish`

In [18]:
def fill_missing_user_id(df: pd.DataFrame,
                         events_df: pd.DataFrame):
  '''
    Attempting to find match that was played against user_id with Nan value (missing_user).
    Its opponent will have same map_id and timestamp, and user_id same as opponent_id stored in missing_user
  '''
  missing_user = df[df['user_id'].isna()].copy()
  ids_to_drop = []

  # TODO pokusaj vektorsku iteraciju umesto iterrows
  for idx, row in missing_user.iterrows():
    opponent_id = row['opponent_id']
    map_id = row['map_id']
    timestamp = row['timestamp']

    # Ukoliko nedostaje podatak, nije moguce jedinstveno upariti igraca, pa ga uklanjamo
    if pd.isna(opponent_id) or pd.isna(map_id):
      ids_to_drop.append(row['id'])
      continue

    partner = df[
      (df['user_id'] == opponent_id) &
      (df['map_id'] == map_id) &
      (df['timestamp'] == timestamp)
    ]

    if len(partner) == 1:
      found_user_id = partner.iloc[0]['opponent_id']
      df.at[idx, 'user_id'] = found_user_id

      # Ukoliko smo nasli odgovarajuci user_id, azuriramo ga u events_df
      events_df.loc[events_df['id'] == row['id'], 'user_id'] = found_user_id
    else:
      # Nasli smo duplikat koji treba da uklonimo
      ids_to_drop.append(row['id'])

  return df, events_df, ids_to_drop

In [19]:
finish_df, events_df, finish_drop = fill_missing_user_id(finish_df, events_df)
start_df, events_df, start_drop = fill_missing_user_id(start_df, events_df)

all_ids_to_drop = set(finish_drop + start_drop)
events_df = events_df[~events_df['id'].isin(all_ids_to_drop)]

print(f"Uklonjenih redova: {len(all_ids_to_drop)}")
print("NaN user_id u events_df:", events_df['user_id'].isna().sum())

Uklonjenih redova: 3

NaN user_id u events_df: 50

Sada su ostali samo session_ping podaci. Za sada cemo ih brisemo

### Removing `session_ping` rows where user_id=NaN

In [20]:
events_df = events_df[
    ~((events_df['event_type'] == 'session_ping') &
      (events_df['user_id'].isna()))
]

## Add username field to `events_df`

In [21]:
unique_user_id = events_df['user_id'].unique()

In [22]:
events_registration = events_df[events_df['event_type']=='registration'].copy()

# Extract desired fields from 'event_data' into new columns
registration_details_df = pd.DataFrame({
    'user_id': events_registration['user_id'],
    'username': events_registration['event_data'].apply(lambda x: x.get('username')),
    'country': events_registration['event_data'].apply(lambda x: x.get('country')),
    'device_os': events_registration['event_data'].apply(lambda x: x.get('device_os'))
})

# Display the new DataFrame
print("New registration details table (first 5 rows):")
print(registration_details_df.head(3))

# Create a mapping from user_id to username from registration_details_df
user_map = registration_details_df.set_index('user_id')['username'].to_dict()

# Add the 'username' column to events_df
events_df['username'] = events_df['user_id'].map(user_map)

print("\nEvents DataFrame with new 'username' column (first 5 rows):")
print(events_df.head(3))

New registration details table (first 5 rows):

user_id         username country device_os
0  5129c6fc-f112-4675-ad99-af28f058a59c    CosmicRay2014     DEU       iOS
8  57dc71fe-00cb-4b2f-8a74-4aa1fb127844  ThunderBolt6169     BIH   Android
9  f79f5ff5-1bf5-4947-89e5-b688afe4030d  OrangeJuice3833     SRB       iOS

Events DataFrame with new 'username' column (first 5 rows):

id   timestamp    event_type                               user_id  \
0  2790  1775171101  registration  5129c6fc-f112-4675-ad99-af28f058a59c   
1  2791  1775174844  session_ping  5129c6fc-f112-4675-ad99-af28f058a59c   
2  2792  1775174964  session_ping  5129c6fc-f112-4675-ad99-af28f058a59c   

                                          event_data       username  
0  {'country': 'DEU', 'device_os': 'iOS', 'userna...  CosmicRay2014  
1       {'state': 'started', 'device_os': 'Android'}  CosmicRay2014  
2   {'state': 'in_progress', 'device_os': 'Android'}  CosmicRay2014

## Add `date` field, convert timestamp to date

In [23]:
events_df['date'] = pd.to_datetime(events_df['timestamp'], unit='s')
print("Added 'date' column to events_df, converted from 'timestamp'.")
print(events_df[['timestamp', 'date']].head(3))

Added 'date' column to events_df, converted from 'timestamp'.

timestamp                date
0  1775171101 2026-04-02 23:05:01
1  1775174844 2026-04-03 00:07:24
2  1775174964 2026-04-03 00:09:24

## Checking missing fields in `event_data` field

In [25]:
# TODO uvedi tip za svaku polje, pa bi lakse proveravao koji tip je validan
Id = NewType('Id', int)
EventType = NewType('EventType', str)

In [26]:
# for given `event_type` returns ids that don't have required fields
missing_event_type_dict: Dict[EventType, List[Id]] = dict()

In [27]:
'''
  Returns True if all values in fields are in the data_dict
'''
def check_required_fields(data_dict: Dict,
                          fields: List) -> bool:
    if not isinstance(data_dict, dict):
        return False # Not a dictionary, so fields can't be present

    for field in fields:
      if field not in data_dict:
        return False
    return True

In [28]:
def fill_missing_fields(fields: Dict,
                        id_list: List[Id]) -> bool:
  '''
    Fill missing field in event_data by None
  '''
  for event_id in id_list:
    selected_id_data = events_df.index[events_df['id'] == event_id].tolist()

    if not selected_id_data:
      continue

    index = selected_id_data[0]

    # pravimo kopiju da izbegnemo probleme sa menjanjem originala
    data = events_df.at[index, 'event_data'].copy()

    updated = False
    for field in fields:
      if field not in data:
        data[field] = None
        updated = True

    if updated:
      events_df.at[index, 'event_data'] = data

  return True

### Checking `registration` `event_type`


We have all regular event_type names

In [29]:
events_registration_df = events_df[events_df['event_type']=='registration'].copy()

In [30]:
events_registration_df.head()

,id,timestamp,event_type,user_id,event_data,username,date
0,2790,1775171101,registration,5129c6fc-f112-4675-ad99-af28f058a59c,"{'country': 'DEU', 'device_os': 'iOS', 'userna...",CosmicRay2014,2026-04-02 23:05:01
8,799,1775176298,registration,57dc71fe-00cb-4b2f-8a74-4aa1fb127844,"{'country': 'BIH', 'device_os': 'Android', 'us...",ThunderBolt6169,2026-04-03 00:31:38
9,636,1775179205,registration,f79f5ff5-1bf5-4947-89e5-b688afe4030d,"{'country': 'SRB', 'device_os': 'iOS', 'userna...",OrangeJuice3833,2026-04-03 01:20:05
16,185,1775179688,registration,bb5f9d21-16d4-4587-938d-53afacf47cb1,"{'country': 'DEU', 'device_os': 'iOS', 'userna...",EmeraldDragon1712,2026-04-03 01:28:08
23,3390,1775180256,registration,c1014f2f-e8bf-48d6-bf24-dbbefc8124a0,"{'country': 'SRB', 'device_os': 'iOS', 'userna...",SolarFlare9591,2026-04-03 01:37:36


In [31]:
required_fields = ['country', 'device_os', 'username']

# Apply the check to the 'event_data' column
events_registration_df['has_required_fields'] = events_registration_df['event_data'].apply(lambda x: check_required_fields(x, required_fields))

# Count how many rows have all required fields
num_rows_with_all_fields = events_registration_df['has_required_fields'].sum()

print(f"Number of registration events with all required fields ({required_fields}): {num_rows_with_all_fields}")
print(f"Total registration events: {len(events_registration_df)}")

# Display some rows to show the new column
print(events_registration_df[['event_data', 'has_required_fields']].head())

Number of registration events with all required fields (['country', 'device_os', 'username']): 48

Total registration events: 50

event_data  has_required_fields
0   {'country': 'DEU', 'device_os': 'iOS', 'userna...                 True
8   {'country': 'BIH', 'device_os': 'Android', 'us...                 True
9   {'country': 'SRB', 'device_os': 'iOS', 'userna...                 True
16  {'country': 'DEU', 'device_os': 'iOS', 'userna...                 True
23  {'country': 'SRB', 'device_os': 'iOS', 'userna...                 True

In [32]:
# Mapping username to country and registration_date
user_details_map = {}

for index, row in events_registration_df.iterrows():
  username = row['event_data'].get('username')
  country = row['event_data'].get('country')
  registration_date = row['date']
  user_details_map[username] = {
    'country': country,
    'registration_date': registration_date
  }

# Displaying first 3 users
from itertools import islice
for k, v in islice(user_details_map.items(), 3):
    print(f"{k}: {v}")

CosmicRay2014: {'country': 'DEU', 'registration_date': Timestamp('2026-04-02 23:05:01')}

ThunderBolt6169: {'country': 'BIH', 'registration_date': Timestamp('2026-04-03 00:31:38')}

OrangeJuice3833: {'country': 'SRB', 'registration_date': Timestamp('2026-04-03 01:20:05')}

In [33]:
# Find rows where 'has_required_fields' is False
false_required_fields_df = events_registration_df[events_registration_df['has_required_fields'] == False]

# Extract the 'id' column and convert it to a list
ids_with_missing_fields = false_required_fields_df['id'].tolist()

print(f"Number of events with missing required registration fields: {len(ids_with_missing_fields)}")
print("IDs of events with missing required fields:")
print(ids_with_missing_fields)

missing_event_type_dict['registration'] = ids_with_missing_fields

Number of events with missing required registration fields: 2

IDs of events with missing required fields:

[1881, 1384]

Filling missing data fields with None

In [34]:
fill_missing_fields(required_fields, ids_with_missing_fields)

True

### Checking `session_ping` `event_type`

In [35]:
events_session_df = events_df[events_df['event_type']=='session_ping'].copy()

In [36]:
events_session_df.head()

,id,timestamp,event_type,user_id,event_data,username,date
1,2791,1775174844,session_ping,5129c6fc-f112-4675-ad99-af28f058a59c,"{'state': 'started', 'device_os': 'Android'}",CosmicRay2014,2026-04-03 00:07:24
2,2792,1775174964,session_ping,5129c6fc-f112-4675-ad99-af28f058a59c,"{'state': 'in_progress', 'device_os': 'Android'}",CosmicRay2014,2026-04-03 00:09:24
3,2793,1775175084,session_ping,5129c6fc-f112-4675-ad99-af28f058a59c,"{'state': 'in_progress', 'device_os': 'Android'}",CosmicRay2014,2026-04-03 00:11:24
4,2794,1775175204,session_ping,5129c6fc-f112-4675-ad99-af28f058a59c,"{'state': 'in_progress', 'device_os': 'Android'}",CosmicRay2014,2026-04-03 00:13:24
5,2795,1775175324,session_ping,5129c6fc-f112-4675-ad99-af28f058a59c,"{'state': 'in_progress', 'device_os': 'Android'}",CosmicRay2014,2026-04-03 00:15:24


In [37]:
required_fields = ['state', 'device_os']

events_session_df['has_required_fields'] = events_session_df['event_data'].apply(lambda x: check_required_fields(x, required_fields))

# Count how many rows have all required fields
num_rows_with_all_fields = events_session_df['has_required_fields'].sum()

print(f"Number of registration events with all required fields: {num_rows_with_all_fields}")
print(f"Total registration events: {len(events_session_df)}")

Number of registration events with all required fields: 3648

Total registration events: 3826

In [38]:
false_required_fields_df = events_session_df[events_session_df['has_required_fields'] == False]

# Extract the 'id' column and convert it to a list
ids_with_missing_fields = false_required_fields_df['id'].tolist()

print(f"Number of events with missing required session_ping fields: {len(ids_with_missing_fields)}")
print("IDs of events with missing required fields:")
print(ids_with_missing_fields)

missing_event_type_dict['session_ping'] = ids_with_missing_fields

Number of events with missing required session_ping fields: 178

IDs of events with missing required fields:

[
    644,
    3400,
    1892,
    1893,
    3811,
    3814,
    526,
    3405,
    804,
    3408,
    532,
    2806,
    2809,
    4754,
    66,
    2032,
    2833,
    2837,
    812,
    663,
    664,
    668,
    389,
    2253,
    3833,
    3838,
    942,
    221,
    2265,
    4110,
    954,
    2165,
    2167,
    2171,
    673,
    427,
    2406,
    4220,
    243,
    3856,
    4116,
    4368,
    88,
    1948,
    551,
    4786,
    1953,
    438,
    440,
    2189,
    1428,
    2863,
    4243,
    560,
    4382,
    4798,
    4800,
    4802,
    1197,
    4133,
    573,
    3719,
    3720,
    3314,
    583,
    4264,
    2518,
    4901,
    970,
    4490,
    4394,
    3985,
    3893,
    3325,
    1108,
    3725,
    3,
    3002,
    3160,
    3006,
    1449,
    4411,
    457,
    3017,
    3471,
    979,
    2941,
    4691,
    4692,
    3026,
    2695,
    1325,
    5075,
    2332,
    4606,
    2535,
    1118,
    859,
    2077,
    986,
    991,
    21,
    23,
    4515,
    3475,
    3477,
    724,
    4161,
    3992,
    4939,
    1668,
    2708,
    2206,
    4532,
    272,
    1122,
    4415,
    4420,
    2548,
    3088,
    2949,
    1540,
    5106,
    1545,
    477,
    998,
    1124,
    112,
    113,
    2438,
    2443,
    3339,
    3341,
    2954,
    883,
    3624,
    2560,
    1475,
    4553,
    3348,
    1988,
    3047,
    1006,
    1819,
    1824,
    3756,
    1990,
    4713,
    1032,
    738,
    740,
    1037,
    2732,
    1253,
    3239,
    1831,
    2561,
    3584,
    3766,
    4429,
    295,
    2568,
    2101,
    2632,
    327,
    611,
    2585,
    1727,
    2891,
    4437,
    1492,
    1013,
    2894,
    45,
    2224,
    3490,
    4577,
    1863
]

In [39]:
fill_missing_fields(required_fields, ids_with_missing_fields)

True

### Checking `match_start` `event_type`

In [40]:
events_match_start_df = events_df[events_df['event_type']=='match_start'].copy()

In [41]:
events_match_start_df.head()

,id,timestamp,event_type,user_id,event_data,username,date
82,5369,1775207066,match_start,031ddc37-6878-4af6-b043-09a5ad23b653,{'map_id': 'fa508820-49dd-472a-b3bb-88931d7c05...,CosmicRay2271,2026-04-03 09:04:26
171,5896,1775234699,match_start,5129c6fc-f112-4675-ad99-af28f058a59c,{'map_id': 'fa508820-49dd-472a-b3bb-88931d7c05...,CosmicRay2014,2026-04-03 16:44:59
172,5895,1775234699,match_start,55046966-a746-4654-9d96-8464f7a8312b,{'map_id': 'fa508820-49dd-472a-b3bb-88931d7c05...,AcidRain5624,2026-04-03 16:44:59
176,5610,1775234810,match_start,5129c6fc-f112-4675-ad99-af28f058a59c,{'map_id': 'db6f0747-d360-4427-be0c-82b78e51f7...,CosmicRay2014,2026-04-03 16:46:50
296,5656,1775252735,match_start,f79f5ff5-1bf5-4947-89e5-b688afe4030d,{'map_id': '42093495-255a-47b5-bdcd-347451b9f3...,OrangeJuice3833,2026-04-03 21:45:35


In [42]:
required_fields = ['map_id', 'opponent_id']

# Apply the check to the 'event_data' column
events_match_start_df['has_required_fields'] = events_match_start_df['event_data'].apply(lambda x: check_required_fields(x, required_fields))

# Count how many rows have all required fields
num_rows_with_all_fields = events_match_start_df['has_required_fields'].sum()

print(f"Number of registration events with all required fields ({', '.join(required_fields)}): {num_rows_with_all_fields}")
print(f"Total registration events: {len(events_match_start_df)}")

Number of registration events with all required fields (map_id, opponent_id): 454

Total registration events: 477

In [43]:
false_required_fields_df = events_match_start_df[events_match_start_df['has_required_fields'] == False]

# Extract the 'id' column and convert it to a list
ids_with_missing_fields = false_required_fields_df['id'].tolist()

print(f"Number of events with missing required match_start fields: {len(ids_with_missing_fields)}")
print("IDs of events with missing required fields:")
print(ids_with_missing_fields)

missing_event_type_dict['match_start'] = ids_with_missing_fields

Number of events with missing required match_start fields: 23

IDs of events with missing required fields:

[
    5251,
    5818,
    5538,
    5397,
    5613,
    5339,
    5785,
    6190,
    5511,
    5884,
    6344,
    5308,
    6151,
    5375,
    5723,
    6398,
    5167,
    5208,
    6166,
    6134,
    5955,
    5557,
    5218
]

In [44]:
fill_missing_fields(required_fields, ids_with_missing_fields)

True

### Checking `match_finish` `event_type`

In [45]:
events_match_finish_df = events_df[events_df['event_type']=='match_finish'].copy()

In [46]:
events_match_finish_df.head()

,id,timestamp,event_type,user_id,event_data,username,date
89,5370,1775207423,match_finish,031ddc37-6878-4af6-b043-09a5ad23b653,{'map_id': 'fa508820-49dd-472a-b3bb-88931d7c05...,CosmicRay2271,2026-04-03 09:10:23
90,5371,1775207423,match_finish,ae034bd0-3925-4259-b641-44d013a77d18,{'map_id': 'fa508820-49dd-472a-b3bb-88931d7c05...,LunarEclipse1737,2026-04-03 09:10:23
178,5897,1775234834,match_finish,55046966-a746-4654-9d96-8464f7a8312b,{'map_id': 'fa508820-49dd-472a-b3bb-88931d7c05...,AcidRain5624,2026-04-03 16:47:14
179,5898,1775234834,match_finish,5129c6fc-f112-4675-ad99-af28f058a59c,{'map_id': 'fa508820-49dd-472a-b3bb-88931d7c05...,CosmicRay2014,2026-04-03 16:47:14
181,5612,1775234933,match_finish,55046966-a746-4654-9d96-8464f7a8312b,{'map_id': 'db6f0747-d360-4427-be0c-82b78e51f7...,AcidRain5624,2026-04-03 16:48:53


In [47]:
required_fields = ['map_id', 'opponent_id', 'outcome']

# Apply the check to the 'event_data' column
events_match_finish_df['has_required_fields'] = events_match_finish_df['event_data'].apply(lambda x: check_required_fields(x, required_fields))

# Count how many rows have all required fields
num_rows_with_all_fields = events_match_finish_df['has_required_fields'].sum()

print(f"Number of registration events with all required fields ({', '.join(required_fields)}): {num_rows_with_all_fields}")
print(f"Total registration events: {len(events_match_finish_df)}")

Number of registration events with all required fields (map_id, opponent_id, outcome): 447

Total registration events: 474

In [48]:
false_required_fields_df = events_match_finish_df[events_match_finish_df['has_required_fields'] == False]

# Extract the 'id' column and convert it to a list
ids_with_missing_fields = false_required_fields_df['id'].tolist()

print(f"Number of events with missing required match_finish fields: {len(ids_with_missing_fields)}")
print("IDs of events with missing required fields:")
print(ids_with_missing_fields)

missing_event_type_dict['match_finish'] = ids_with_missing_fields

Number of events with missing required match_finish fields: 27

IDs of events with missing required fields:

[
    5195,
    5388,
    6146,
    6383,
    6069,
    6192,
    6266,
    6154,
    5503,
    6085,
    5458,
    6055,
    5556,
    5315,
    5850,
    6066,
    5360,
    6374,
    5188,
    6270,
    5407,
    5148,
    6160,
    6357,
    5558,
    6058,
    5744
]

In [49]:
fill_missing_fields(required_fields, ids_with_missing_fields)

True

### Removing uncomplete `registration` and `session_ping` fields (optional)

Necemo da uklanjamo ove podatke, jer smo tamo gde nedostaje polje upisali None

In [50]:
'''
registration_ids = missing_event_type_dict['registration']
session_ping_ids = missing_event_type_dict['session_ping']

ids_to_remove = registration_ids + session_ping_ids

events_df = events_df[~events_df['id'].isin(ids_to_remove)]

print(f"Removed {len(ids_to_remove)} rows from events_df.")
print(f"New events_df shape: {events_df.shape}")
'''

'\nregistration_ids = missing_event_type_dict[\'registration\']\nsession_ping_ids = missing_event_type_dict[\'session_ping\']\n\nids_to_remove = registration_ids + session_ping_ids\n\nevents_df = events_df[~events_df[\'id\'].isin(ids_to_remove)]\n\nprint(f"Removed {len(ids_to_remove)} rows from events_df.")\nprint(f"New events_df shape: {events_df.shape}")\n'

### Analyzing `match_start` and `match_finish` uncomplete fields

`missing_event_type_dict` je recnik koji za dati `event_type` vraca listu `id` gde imamo nepotpune podatke

In [51]:
match_start_ids = missing_event_type_dict['match_start']
match_finish_ids = missing_event_type_dict['match_finish']

Prvo gledamo `match_start` podatke

#### Analyzing `match_start` fields

Gledamo koja polja nedostaju



In [52]:
for id in match_start_ids:
  p=events_df[events_df['id']==id]
  print(p['event_data'].iloc[0])

{'map_id': '42093495-255a-47b5-bdcd-347451b9f3bf', 'opponent_id': None}

{'opponent_id': '84304216-13ec-4d6c-ad86-ef964b7ecd30', 'map_id': None}

{'opponent_id': '4c800e05-5cfd-4884-a022-816d0ed5eb21', 'map_id': None}

{'map_id': 'fa508820-49dd-472a-b3bb-88931d7c059f', 'opponent_id': None}

{'map_id': 'db6f0747-d360-4427-be0c-82b78e51f707', 'opponent_id': None}

...

{'opponent_id': '727ea47d-d720-419c-a82b-3c63ad149888', 'map_id': None}

{'opponent_id': '6d2b2743-af93-4df3-8f5a-2192032f40c9', 'map_id': None}

{'opponent_id': 'afd79b8a-24db-4c4f-96e9-d33d2c57639d', 'map_id': None}

{'map_id': 'fa508820-49dd-472a-b3bb-88931d7c059f', 'opponent_id': None}

{'map_id': 'db6f0747-d360-4427-be0c-82b78e51f707', 'opponent_id': None}

Ne mozemo samo sa jednim od ova 2 podatka da nadjemo tacno ko nam je bio protivnik, pa ih uklanjamo

In [53]:
events_df = events_df[~events_df['id'].isin(match_start_ids)]

#### Analyzing `match_finish` fields

Sada gledamo match_finish podatke. Ako nadjemo mec gde ima `map_id` ako i `opponent_id`, a fali samo `outcome` onda bi to moglo da sracuna

In [54]:
for id in match_finish_ids:
  p = events_df[events_df['id']==id]
  event_data = p['event_data'].iloc[0]
  if 'map_id' in event_data and 'opponent_id' in event_data:
    print(f"{id = }")
    print(f"timestamp = {p['timestamp'].iloc[0]}")
    print(event_data)

id = 5195

timestamp = 1775300795

{
    'map_id': '42093495-255a-47b5-bdcd-347451b9f3bf',
    'opponent_id': 'e6fcde77-13b9-41ef-8eba-15657f54e0cb',
    'outcome': None
}

id = 5388

timestamp = 1775307969

...

timestamp = 1775513802

{'opponent_id': '1f7cdf89-e4e3-4935-8412-b27f2cfc588c', 'outcome': 1.0, 'map_id': None}

id = 5744

timestamp = 1775521583

{'opponent_id': 'e6fcde77-13b9-41ef-8eba-15657f54e0cb', 'outcome': 1.0, 'map_id': None}

TESTING Proveramo da li postoji podatak za uneti `user_id` i `timestamp`

In [55]:
specific_user_id = '8e9525f8-1ffd-4aae-b7f7-d55bcbf2b089'
#specific_map_id = '42093495-255a-47b5-bdcd-347451b9f3bf'
specific_timestamp = 1775504985

filtered_events = events_df[
    (events_df['event_type'] == 'match_finish') &
    (events_df['user_id'] == specific_user_id) &
    (events_df['timestamp'] == specific_timestamp) #&
    #(events_df['event_data'].apply(lambda x: x.get('map_id') == specific_map_id))
]

print(f"Found {len(filtered_events)} events matching the criteria.")
if not filtered_events.empty:
    print("Matching events:")
    print(filtered_events)

Found 1 events matching the criteria.

Matching events:

id   timestamp    event_type                               user_id  \
4400  6356  1775504985  match_finish  8e9525f8-1ffd-4aae-b7f7-d55bcbf2b089   

                                             event_data        username  \
4400  {'map_id': '42093495-255a-47b5-bdcd-347451b9f3...  SolarFlare9264   

                    date  
4400 2026-04-06 19:49:45

### Interpolating data from same `match_finish` events

In [56]:
# Uzimamo dataframe of svih match_finish kojima nedostaje neko polje
incomplete_finish = events_df[events_df['id'].isin(match_finish_ids)]
outcome_inverse = {1: 0, 0: 1, 0.5: 0.5}  # za racunanje outcome od protivnika
ids_to_drop = list()

for idx, row in incomplete_finish.iterrows():
  id = row['id']
  user_id = row['user_id']
  opponent_id = row['event_data'].get('opponent_id')
  timestamp = row['timestamp']

  # Pronađi match_finish od opponent-a sa istim timestamp-om
  partner = events_match_finish_df[
    (events_match_finish_df['user_id'] == opponent_id) &
    (events_match_finish_df['timestamp'] == timestamp)
  ]

  if len(partner) == 1:
    partner_data = partner.iloc[0]['event_data']

    # Posto smo nasli protivnika, popunjavamo polja koja mu nedostaju
    for field in ['map_id', 'opponent_id']:
      if events_df.at[idx, 'event_data'].get(field) is None:
        value = partner_data.get(field)
        if value:
          events_df.at[idx, 'event_data'][field] = value
          print(f"{idx = }, '{field}' added at  {value = }")
        else:
          print(f"{idx = }, '{field}' error at  {value = }")

    # Obrada outcome
    if events_df.at[idx, 'event_data'].get('outcome') is None:
      partner_outcome = partner_data.get('outcome')
      if partner_outcome:
        events_df.at[idx, 'event_data']['outcome'] = outcome_inverse[partner_outcome]
        print(f"{idx = }, 'outcome' added at  {partner_outcome = }")
      else:
          print(f"{idx = }, 'outcome' error at  {partner_outcome = }")
  else:
    # Ukoliko nismo nasli protivnika ili smo naisli na duplikat, uklanjamo podatak
    print(f"| {row['id'] = } Found {partner} at ")
    ids_to_drop.append(row['id'])

idx = 736, 'outcome' added at  partner_outcome = 0.5

idx = 806, 'map_id' added at  value = '42093495-255a-47b5-bdcd-347451b9f3bf'

| row['id'] = 6146 Found Empty DataFrame
Columns: 
Index: [] at

idx = 1175, 'map_id' added at  value = 'fa508820-49dd-472a-b3bb-88931d7c059f'

idx = 1217, 'outcome' added at  partner_outcome = 0.5

...

| row['id'] = 6160 Found Empty DataFrame
Columns: 
Index: [] at

idx = 4401, 'outcome' added at  partner_outcome = 0.5

idx = 4520, 'outcome' error at  partner_outcome = 0.0

idx = 4626, 'map_id' added at  value = 'db6f0747-d360-4427-be0c-82b78e51f707'

idx = 4800, 'map_id' added at  value = 'fa508820-49dd-472a-b3bb-88931d7c059f'

### Removing `match_finish` data that does not have a partner

In [57]:
events_df = events_df[~events_df['id'].isin(ids_to_drop)]

print(f"Removed {len(ids_to_drop)} rows from events_df.")
print(f"New events_df shape: {events_df.shape}")

Removed 7 rows from events_df.

New events_df shape: (4797, 7)

## Monitoring `session_ping` intervals (In progress)

In [58]:
# Koliko vremena prolazi izmedju 2 session_pinga, moglo bi da bude korisno da bismo pratili device_os
u_ids=registration_details_df['user_id'].tolist()
for id in u_ids:
  df=events_df[events_df['user_id']==id]
  df=df[df['event_type']=='session_ping']
  #print(df['timestamp'].diff())

## Checking complete games

In [62]:
def get_standardized_events(df: pd.DataFrame,
                            event_type: str):
  subset = df[df['event_type'] == event_type].copy()

  subset['opponent_id'] = subset['event_data'].apply(lambda x: x.get('opponent_id'))
  subset['map_id'] = subset['event_data'].apply(lambda x: x.get('map_id'))

  # Normalizujemo igrace da bismo izbegli duplikate (p1 je uvek "manji" string, p2 "veći")
  subset['p1'] = subset.apply(lambda x: min(str(x['user_id']), str(x['opponent_id'])), axis=1)
  subset['p2'] = subset.apply(lambda x: max(str(x['user_id']), str(x['opponent_id'])), axis=1)

  # Kreiramo kolonu za vreme kako se ne bi izgubila u merge_asof
  if event_type == 'match_start':
    subset['timestamp_start'] = subset['timestamp']
  else:
    subset['timestamp_finish'] = subset['timestamp']

    # Logika za ishod
    def calculate_p1_outcome(row):
      val = row['event_data'].get('outcome')
      if val is None: return None

      val = float(val) # Osiguravamo da je float (0.0, 1.0 ili 0.5)

      # Ako je log poslao igrač p1, uzmi njegov outcome direktno
      if str(row['user_id']) == str(row['p1']):
        return val
      # Ako je log poslao igrač p2, ishod za p1 je (1 - ishod_p2)
      else:
        return 1.0 - val

    subset['outcome_p1'] = subset.apply(calculate_p1_outcome, axis=1)

  # Uklanjanje duplikata
  group_cols = ['p1', 'p2', 'map_id', 'timestamp']
  if event_type == 'match_finish':
    standardized = subset.groupby(group_cols).agg({
      'timestamp_finish': 'first',
      'outcome_p1': 'first'
    }).reset_index()
  else:
    # onda je poslay 'match_start'
    standardized = subset.groupby(group_cols).agg({
      'timestamp_start': 'first'
    }).reset_index()

  return standardized.sort_values('timestamp')

Pomocu merge_asof cemo moci da spajamo meceve za koje jedan igrac ima match_start a drugi match_finish zato sto cemo spajanje raditi preko najblizeg timestamp

In [65]:
clean_start = get_standardized_events(events_df, 'match_start')
clean_finish = get_standardized_events(events_df, 'match_finish')

# merge_asof koristi 'timestamp' za uparivanje, ali će zadržati timestamp_start i timestamp_finish
complete_matches = pd.merge_asof(
    clean_finish,
    clean_start[['p1', 'p2', 'map_id', 'timestamp', 'timestamp_start']],
    on='timestamp',
    by=['p1', 'p2', 'map_id'],
    direction='backward'
)

# apiranje username-ova
complete_matches['username'] = complete_matches['p1'].map(user_map)
complete_matches['opponent_username'] = complete_matches['p2'].map(user_map)

# Sređivanje kolona
# database_df ce nam biti dataframe sa svim odigranim mecevima
database_df = complete_matches[[
    'username',
    'opponent_username',
    'map_id',
    'timestamp_start',
    'timestamp_finish',
    'outcome_p1'
]].copy()

# Preimenovanje za bazu
database_df.columns = ['username', 'opponent_username', 'map_id', 'date_start', 'date_finish', 'outcome']

# Ukloni mečeve bez početka
database_df = database_df.dropna(subset=['date_start'])

# Konverzija formata vremena (ako su timestampovi bili datetime objekti)
# Ako su već bili stringovi ili brojevi, pd.to_datetime će ih srediti
database_df['date_start'] = pd.to_datetime(database_df['date_start'], unit='s').dt.strftime('%Y-%m-%d %H:%M:%S')
database_df['date_finish'] = pd.to_datetime(database_df['date_finish'], unit='s').dt.strftime('%Y-%m-%d %H:%M:%S')

## Creating databases

Formiranje baze

In [66]:
# Čuvanje
conn = sqlite3.connect('match_history.db')
database_df.to_sql('matches', conn, if_exists='replace', index=False)
conn.close()

print(f"Uspešno sačuvano {len(database_df)} mečeva u bazu.")
display(database_df.head())

Uspešno sačuvano 256 mečeva u bazu.

,username,opponent_username,map_id,date_start,date_finish,outcome
0,CosmicRay2271,LunarEclipse1737,fa508820-49dd-472a-b3bb-88931d7c059f,2026-04-03 09:04:26,2026-04-03 09:10:23,1.0
1,CosmicRay2014,AcidRain5624,fa508820-49dd-472a-b3bb-88931d7c059f,2026-04-03 16:44:59,2026-04-03 16:47:14,0.0
2,CosmicRay2014,AcidRain5624,db6f0747-d360-4427-be0c-82b78e51f707,2026-04-03 16:46:50,2026-04-03 16:48:53,1.0
3,EmeraldDragon1712,OrangeJuice3833,42093495-255a-47b5-bdcd-347451b9f3bf,2026-04-03 21:45:35,2026-04-03 21:47:35,1.0
5,ThunderBolt6169,AcidRain5106,89e9ef59-d7df-459a-b558-52b00d5b8f91,2026-04-04 00:34:54,2026-04-04 00:36:56,0.0


## Filling database for API

In [68]:
# Mapiranje `id` mape na njen naziv
map_id_to_name = maps_df.set_index('id')['name'].to_dict()

database_df['map_name'] = database_df['map_id'].map(map_id_to_name)

# Convert 'date_start' and 'date_finish' back to datetime objects for calculation
database_df['date_start_dt'] = pd.to_datetime(database_df['date_start'])
database_df['date_finish_dt'] = pd.to_datetime(database_df['date_finish'])

# Calculate duration and convert to total seconds
database_df['duration_seconds'] = (database_df['date_finish_dt'] - database_df['date_start_dt']).dt.total_seconds()

# Drop the temporary datetime columns if they are not needed further
database_df = database_df.drop(columns=['date_start_dt', 'date_finish_dt'])

print("Database DataFrame updated with 'map_name' and 'duration_seconds' columns:")
display(database_df.head())

Database DataFrame updated with 'map_name' and 'duration_seconds' columns:

,username,opponent_username,map_id,date_start,date_finish,outcome,map_name,duration_seconds
0,CosmicRay2271,LunarEclipse1737,fa508820-49dd-472a-b3bb-88931d7c059f,2026-04-03 09:04:26,2026-04-03 09:10:23,1.0,Cobblestone,357.0
1,CosmicRay2014,AcidRain5624,fa508820-49dd-472a-b3bb-88931d7c059f,2026-04-03 16:44:59,2026-04-03 16:47:14,0.0,Cobblestone,135.0
2,CosmicRay2014,AcidRain5624,db6f0747-d360-4427-be0c-82b78e51f707,2026-04-03 16:46:50,2026-04-03 16:48:53,1.0,Inferno,123.0
3,EmeraldDragon1712,OrangeJuice3833,42093495-255a-47b5-bdcd-347451b9f3bf,2026-04-03 21:45:35,2026-04-03 21:47:35,1.0,Forest,120.0
5,ThunderBolt6169,AcidRain5106,89e9ef59-d7df-459a-b558-52b00d5b8f91,2026-04-04 00:34:54,2026-04-04 00:36:56,0.0,Desert,122.0


In [69]:
# Sada cemo podeliti podatke na primarne igrače i protivnike
# p1 ce biti primarni igraci
p1_data = database_df[['username', 'map_name', 'outcome']].copy()
p1_data.columns = ['username', 'map_name', 'outcome']

# p2 ce biti protivnici
# oni dobijaju suprotan 'outcome'
p2_data = database_df[['opponent_username', 'map_name', 'outcome']].copy()
p2_data['outcome'] = 1.0 - p2_data['outcome']
p2_data = p2_data[['opponent_username', 'map_name', 'outcome']]
p2_data.columns = ['username', 'map_name', 'outcome']

# Sada spajamo sve igrace, i to predstavlja sve odigrane meceve
all_results = pd.concat([p1_data, p2_data], ignore_index=True)

# Formiramo bazu igraca i mapa, uz nove kolone za total_outcome i matches_played
database_user_maps = all_results.groupby(['username', 'map_name']).agg(
    total_outcome=('outcome', 'sum'),
    matches_played=('outcome', 'count')
).reset_index()

# Racunamo win ratio za svakog igraca
database_user_maps['win_ratio'] = database_user_maps['total_outcome'] / database_user_maps['matches_played']

# Zaokruzujemo na 2 decimale radi preglednosti
database_user_maps['win_ratio'] = database_user_maps['win_ratio'].round(2)

# opciono, Sortiracemo prema username od igraca
database_user_maps = database_user_maps.sort_values(by=['username', 'total_outcome'], ascending=[True, False])

print("Statistika igraca na odigranim mapama:")
display(database_user_maps.head(10))

Statistika igraca na odigranim mapama:

,username,map_name,total_outcome,matches_played,win_ratio
1,AcidRain4215,Forest,3.5,5,0.70
0,AcidRain4215,Desert,2.0,2,1.00
2,AcidRain4215,Inferno,1.5,2,0.75
3,AcidRain4215,Lake,0.0,2,0.00
4,AcidRain5106,Cobblestone,2.0,5,0.40
5,AcidRain5106,Desert,2.0,4,0.50
6,AcidRain5106,Forest,1.5,2,0.75
7,AcidRain5624,Cobblestone,3.0,4,0.75
10,AcidRain5624,Inferno,1.0,3,0.33
11,AcidRain5624,Lake,1.0,1,1.00


In [72]:
'''
  Get user coutry or registration date, based on input field
'''
def get_user_info(name: str,
                  field: str):
  # Traži korisnika u mapi, ako ga nema vrati default vrednost
  user_data = user_details_map.get(name)
  if not user_data:
    return 'Unknown' if field == 'country' else 'N/A'

  val = user_data.get(field)

  if field == 'registration_date' and pd.notnull(val):
    return val.strftime('%Y-%m-%d')

  return val

In [73]:
# Sortiramo po korisniku, pa po win_ratio (opadajuće) i po broju mečeva (opadajuće)
# Ovo osigurava da ako postoji isti win_ratio, izaberemo mapu koju je više puta igrao
fav_map_df = database_user_maps.sort_values(
    by=['username', 'win_ratio', 'matches_played'],
    ascending=[True, False, False]
)

# Uzimamo samo prvi red za svakog korisnika (onu sa najboljim učinkom)
fav_map_summary = fav_map_df.groupby('username').first().reset_index()
fav_map_summary = fav_map_summary[['username', 'map_name', 'win_ratio']]
fav_map_summary.columns = ['username', 'fav_map', 'fav_map_win_ratio']

# Moramo sabrati duration_seconds i za 'username' i za 'opponent_username' kolone
p1_time = database_df.groupby('username')['duration_seconds'].sum().reset_index()
p1_time.columns = ['user', 'time']

p2_time = database_df.groupby('opponent_username')['duration_seconds'].sum().reset_index()
p2_time.columns = ['user', 'time']

# Spajamo obe uloge i sabiramo ukupno vreme po korisniku
total_playtime_df = pd.concat([p1_time, p2_time]).groupby('user')['time'].sum().reset_index()
total_playtime_df.columns = ['username', 'total_playtime']


# Sada racunamo ukupan win ratio  za svakog igraca
global_stats = database_user_maps.groupby('username').agg(
    total_sum_outcome=('total_outcome', 'sum'),
    total_sum_matches=('matches_played', 'sum')
).reset_index()

global_stats['total_win_ratio'] = (global_stats['total_sum_outcome'] / global_stats['total_sum_matches']).round(2)

# Formiranje krajnjeg dataframe
# fomiramo ga od vise tabela koje spajamo
database_user_summary = fav_map_summary.merge(total_playtime_df, on='username', how='left')
database_user_summary = database_user_summary.merge(global_stats[['username', 'total_win_ratio', 'total_sum_matches']], on='username', how='left')

# Dodavanje default vrednosti u country i registration_date kolone
database_user_summary['country'] = 'Unknown'
database_user_summary['registration_date'] = 'N/A'

database_user_summary['country'] = database_user_summary['username'].apply(lambda x: get_user_info(x, 'country'))
database_user_summary['registration_date'] = database_user_summary['username'].apply(lambda x: get_user_info(x, 'registration_date'))

# Biramo raspored kolona
database_user_summary = database_user_summary[[
    'username', 'country', 'fav_map', 'fav_map_win_ratio',
    'total_playtime', 'total_win_ratio', 'registration_date'
]]

print("Database summary:")
display(database_user_summary.head())

Database summary:

,username,country,fav_map,fav_map_win_ratio,total_playtime,total_win_ratio,registration_date
0,AcidRain4215,GBR,Desert,1.00,2063.0,0.64,2026-04-05
1,AcidRain5106,HRV,Forest,0.75,1816.0,0.50,2026-04-03
2,AcidRain5624,MNE,Lake,1.00,2082.0,0.46,2026-04-03
3,AcidRain9779,GBR,Lake,0.75,2966.0,0.54,2026-04-03
4,BlackPanther5218,SRB,Desert,0.60,2713.0,0.50,2026-04-04


kreiramo sada tabelu za drugi api

In [74]:
df_tmp = database_df.copy()
df_tmp['date_only'] = pd.to_datetime(df_tmp['date_start']).dt.date

# Grupišemo po datumu i nazivu mape
map_daily_stats = df_tmp.groupby(['date_only', 'map_name']).agg(
    avg_playtime=('duration_seconds', 'mean'),
    match_count=('duration_seconds', 'count')
).reset_index()

# Trazimo kumulativno najboljeg igraca
all_dates = sorted(df_tmp['date_only'].unique())
all_maps = df_tmp['map_name'].unique()

cumulative_best = []

for date in all_dates:
    for map_name in all_maps:
        # Kumulativno - svi mečevi do i uključujući ovaj datum
        mask = (
            (df_tmp['map_name'] == map_name) &
            (df_tmp['date_only'] <= date)
        )
        cumulative_df = df_tmp[mask]

        if len(cumulative_df) == 0:
            continue

        # Perspektiva p1
        p1 = cumulative_df[['username', 'outcome']].copy()
        p1.columns = ['user', 'outcome']

        # Perspektiva p2
        p2 = cumulative_df[['opponent_username', 'outcome']].copy()
        p2['outcome'] = 1.0 - p2['outcome']
        p2 = p2[['opponent_username', 'outcome']]
        p2.columns = ['user', 'outcome']

        all_players = pd.concat([p1, p2], ignore_index=True)

        # win ratio = outcome / broj mečeva
        win_ratio = all_players.groupby('user').agg(
            total_outcome=('outcome', 'sum'),
            total_matches=('outcome', 'count')
        )
        win_ratio['win_ratio'] = win_ratio['total_outcome'] / win_ratio['total_matches']
        best_player = win_ratio['win_ratio'].idxmax()

        cumulative_best.append({
            'date_only': str(date),
            'map_name': map_name,
            'best_player_username': best_player
        })

cumulative_best_df = pd.DataFrame(cumulative_best)

map_daily_stats['date_only'] = map_daily_stats['date_only'].astype(str)

# Formiramo bazu sa svim podacima
database_daily_map_leaders = map_daily_stats.merge(
    cumulative_best_df,
    on=['date_only', 'map_name'],
    how='left'
)

database_daily_map_leaders = database_daily_map_leaders[[
    'date_only',
    'map_name',
    'best_player_username',
    'avg_playtime',
    'match_count'
]]

# Konverzija datuma u string radi kompatibilnosti sa bazom
database_daily_map_leaders['date_only'] = database_daily_map_leaders['date_only'].astype(str)

# Zaokruzivanje radi preglednijeg prikaza
database_daily_map_leaders['avg_playtime'] = database_daily_map_leaders['avg_playtime'].round(2)

# Cuvanje u bazu
conn = sqlite3.connect('match_history.db')
database_daily_map_leaders.to_sql('daily_map_leaders', conn, if_exists='replace', index=False)
conn.close()

print("Tabela 'daily_map_leaders' je uspesno kreirana i sacuvana.")
display(database_daily_map_leaders.head())

Tabela 'daily_map_leaders' je uspesno kreirana i sacuvana.

,date_only,map_name,best_player_username,avg_playtime,match_count
0,2026-04-03,Cobblestone,AcidRain5624,246.00,2
1,2026-04-03,Forest,EmeraldDragon1712,120.00,1
2,2026-04-03,Inferno,CosmicRay2014,123.00,1
3,2026-04-04,Cobblestone,AcidRain5624,192.42,12
4,2026-04-04,Desert,CyberNinja9638,183.40,10


## API

In [76]:
app = Flask(__name__)

# Formiranje GET metoda
@app.route('/user-stats', methods=['GET'])
def get_user_stats():
  countries_param = request.args.get('countries')
  df_response = database_user_summary.copy()

  if countries_param:
    countries_list = [c.strip() for c in countries_param.split(',')]
    df_response = df_response[df_response['country'].isin(countries_list)]

  return jsonify(df_response.to_dict(orient='records'))

@app.route('/map-stats/<map_name>', methods=['GET'])
def get_map_stats(map_name):
  date_from = request.args.get('date_from')
  date_to = request.args.get('date_to')

  conn = sqlite3.connect('match_history.db')

  query = "SELECT * FROM daily_map_leaders WHERE map_name = ?"
  params = [map_name]

  if date_from:
    query += " AND date_only >= ?"
    params.append(date_from)
  if date_to:
    query += " AND date_only <= ?"
    params.append(date_to)

  query += " ORDER BY date_only DESC"

  df = pd.read_sql_query(query, conn, params=params)
  conn.close()

  if len(df) == 0:
    return jsonify([])

  df = df.rename(columns={
    'date_only': 'date',
    'match_count': 'match_cnt'
  })

  return jsonify(df[['date', 'avg_playtime', 'best_player_username', 'match_cnt']].to_dict(orient='records'))

@app.route('/maps', methods=['GET'])
def get_maps():
  conn = sqlite3.connect('match_history.db')
  df = pd.read_sql_query("SELECT DISTINCT map_name FROM daily_map_leaders", conn)
  conn.close()
  return jsonify(df['map_name'].tolist())

def run_app():
  app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

Formiramo niti da bismo mogli da istovremeno izvrsavamo i pozive api-ja koriscenjem `requests`

In [77]:
thread = threading.Thread(target=run_app)
thread.daemon = True
thread.start()

## Testiranje API

In [80]:
# Bez filtera
print("-- All users --")
response = requests.get("http://localhost:5000/user-stats")
print(response.json()[:2])

# Testiranje sa filterom za zemlje
print("-- Only SRB and DEU --")
params = {'countries': 'SRB,DEU'}
response = requests.get("http://localhost:5000/user-stats", params=params)
print(response.json())

-- All users --

INFO:werkzeug:127.0.0.1 - - [03/May/2026 17:54:11] "GET /user-stats HTTP/1.1" 200 -


[
    {
        'country': 'GBR',
        'fav_map': 'Desert',
        'fav_map_win_ratio': 1.0,
        'registration_date': '2026-04-05',
        'total_playtime': 2063.0,
        'total_win_ratio': 0.64,
        'username': 'AcidRain4215'
    },
    {
        'country': 'HRV',
        'fav_map': 'Forest',
        'fav_map_win_ratio': 0.75,
        'registration_date': '2026-04-03',
        'total_playtime': 1816.0,
        'total_win_ratio': 0.5,
        'username': 'AcidRain5106'
    }
]

-- Only SRB and DEU --

INFO:werkzeug:127.0.0.1 - - [03/May/2026 17:54:11] "GET /user-stats?countries=SRB,DEU HTTP/1.1" 200 -


[
    {
        'country': 'SRB',
        'fav_map': 'Desert',
        'fav_map_win_ratio': 0.6,
        'registration_date': '2026-04-04',
        'total_playtime': 2713.0,
        'total_win_ratio': 0.5,
        'username': 'BlackPanther5218'
    },
    {
        'country': 'DEU',
        'fav_map': 'Cobblestone',
        'fav_map_win_ratio': 1.0,
        'registration_date': '2026-04-06',
        'total_playtime': 2854.0,
        'total_win_ratio': 0.5,
        'username': 'CodeBreaker4266'
    },
    {
        'country': 'DEU',
        'fav_map': 'Inferno',
        'fav_map_win_ratio': 1.0,
        'registration_date': '2026-04-02',
        'total_playtime': 2781.0,
        'total_win_ratio': 0.42,
        'username': 'CosmicRay2014'
    },
    {
        'country': 'SRB',
        'fav_map': 'Cobblestone',
        'fav_map_win_ratio': 1.0,
        'registration_date': '2026-04-04',
        'total_playtime': 3239.0,
        'total_win_ratio': 0.61,
        'username': 'DragonSlayer1150'
    },
    {
        'country': 'DEU',
        'fav_map': 'Desert',
        'fav_map_win_ratio': 0.67,
        'registration_date': '2026-04-03',
        'total_playtime': 2016.0,
        'total_win_ratio': 0.39,
        'username': 'EmeraldDragon1712'
    },
    {
        'country': 'DEU',
        'fav_map': 'Cobblestone',
        'fav_map_win_ratio': 1.0,
        'registration_date': '2026-04-05',
        'total_playtime': 3042.0,
        'total_win_ratio': 0.64,
        'username': 'EmeraldDragon6728'
    },
    {
        'country': 'SRB',
        'fav_map': 'Cobblestone',
        'fav_map_win_ratio': 1.0,
        'registration_date': '2026-04-03',
        'total_playtime': 2402.0,
        'total_win_ratio': 0.5,
        'username': 'OrangeJuice3833'
    },
    {
        'country': 'SRB',
        'fav_map': 'Lake',
        'fav_map_win_ratio': 1.0,
        'registration_date': '2026-04-04',
        'total_playtime': 2729.0,
        'total_win_ratio': 0.81,
        'username': 'PurpleHaze6693'
    },
    {
        'country': 'DEU',
        'fav_map': 'Inferno',
        'fav_map_win_ratio': 0.67,
        'registration_date': '2026-04-04',
        'total_playtime': 1574.0,
        'total_win_ratio': 0.29,
        'username': 'PurpleHaze9936'
    },
    {
        'country': 'DEU',
        'fav_map': 'Desert',
        'fav_map_win_ratio': 1.0,
        'registration_date': '2026-04-05',
        'total_playtime': 1927.0,
        'total_win_ratio': 0.25,
        'username': 'SolarFlare8869'
    },
    {
        'country': 'DEU',
        'fav_map': 'Inferno',
        'fav_map_win_ratio': 1.0,
        'registration_date': '2026-04-06',
        'total_playtime': 664.0,
        'total_win_ratio': 0.62,
        'username': 'SolarFlare9264'
    },
    {
        'country': 'SRB',
        'fav_map': 'Desert',
        'fav_map_win_ratio': 0.83,
        'registration_date': '2026-04-03',
        'total_playtime': 2120.0,
        'total_win_ratio': 0.55,
        'username': 'SolarFlare9591'
    },
    {
        'country': 'DEU',
        'fav_map': 'Desert',
        'fav_map_win_ratio': 1.0,
        'registration_date': '2026-04-04',
        'total_playtime': 1207.0,
        'total_win_ratio': 0.57,
        'username': 'VaporWave2717'
    }
]

In [84]:
# Showing data for Cobblestone map
response = requests.get('http://localhost:5000/map-stats/Cobblestone')
print(response.json())

# Showing data for Cobblestone map between 03.04. and 04.04.
response = requests.get('http://localhost:5000/map-stats/Cobblestone?date_from=2026-04-03&date_to=2026-04-04')
print(response.json())

# Checking if CS 1.6 map exists here :)
response = requests.get('http://localhost:5000/map-stats/Dust2')
print(response.json())

INFO:werkzeug:127.0.0.1 - - [03/May/2026 17:56:34] "GET /map-stats/Cobblestone HTTP/1.1" 200 -


[
    {'avg_playtime': 251.5, 'best_player_username': 'CodeBreaker4266', 'date': '2026-04-07', 'match_cnt': 2},
    {'avg_playtime': 184.24, 'best_player_username': 'CodeBreaker4266', 'date': '2026-04-06', 'match_cnt': 17},
    {'avg_playtime': 196.29, 'best_player_username': 'AcidRain9779', 'date': '2026-04-05', 'match_cnt': 14},
    {'avg_playtime': 192.42, 'best_player_username': 'AcidRain5624', 'date': '2026-04-04', 'match_cnt': 12},
    {'avg_playtime': 246.0, 'best_player_username': 'AcidRain5624', 'date': '2026-04-03', 'match_cnt': 2}
]

INFO:werkzeug:127.0.0.1 - - [03/May/2026 17:56:34] "GET /map-stats/Cobblestone?date_from=2026-04-03&date_to=2026-04-04 HTTP/1.1" 200 -


[
    {'avg_playtime': 192.42, 'best_player_username': 'AcidRain5624', 'date': '2026-04-04', 'match_cnt': 12},
    {'avg_playtime': 246.0, 'best_player_username': 'AcidRain5624', 'date': '2026-04-03', 'match_cnt': 2}
]

INFO:werkzeug:127.0.0.1 - - [03/May/2026 17:56:34] "GET /map-stats/Dust2 HTTP/1.1" 200 -


[]

## Visualization

In [88]:
# Uzmimamo sve mape koje postoje
maps_response = requests.get("http://localhost:5000/maps")
all_maps = maps_response.json()

# Gledamo period od prethodnih 7 dana od postojecih podataka
available_dates = sorted(database_daily_map_leaders['date_only'].unique())
last_7_dates = available_dates[-7:]

date_from = last_7_dates[0]
date_to = last_7_dates[-1]

rows = []
for map_name in all_maps:
    response = requests.get(
        f"http://localhost:5000/map-stats/{map_name}",
        params={'date_from': date_from, 'date_to': date_to}
    )
    data = response.json()
    for entry in data:
        entry['map_name'] = map_name
        rows.append(entry)

df_chart = pd.DataFrame(rows)

# Vizualizacija
fig = px.line(
    df_chart,
    x='date',
    y='match_cnt',
    color='map_name',
    markers=True,
    title=f'Match counter from {date_from} to {date_to}',
    labels={
        'date': 'Date',
        'match_cnt': 'Number of matches',
        'map_name': 'Map'
    }
)

fig.update_layout(
    xaxis_tickangle=-45,
    legend_title_text='Map',
    hovermode='x unified'
)

fig.show()

INFO:werkzeug:127.0.0.1 - - [03/May/2026 18:01:58] "GET /maps HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/May/2026 18:01:58] "GET /map-stats/Cobblestone?date_from=2026-04-03&date_to=2026-04-07 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/May/2026 18:01:58] "GET /map-stats/Forest?date_from=2026-04-03&date_to=2026-04-07 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/May/2026 18:01:58] "GET /map-stats/Inferno?date_from=2026-04-03&date_to=2026-04-07 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/May/2026 18:01:58] "GET /map-stats/Desert?date_from=2026-04-03&date_to=2026-04-07 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/May/2026 18:01:58] "GET /map-stats/Lake?date_from=2026-04-03&date_to=2026-04-07 HTTP/1.1" 200 -
